In [6]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from prophet import Prophet

# configuration
FRAMES_DIR = "../data/atrium_data/atrium_frames"
DATA_DIR = "../data/new_data"
GRAPH_DIR = "../visualization/graphs"
MODEL_PATH = "yolov8n-seg.pt"

FUTURE_FRAMES = 8000
CONF_THRESHOLD = 0.5

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(GRAPH_DIR, exist_ok=True)

model = YOLO("yolov8n-seg.pt")

# sorting frames
frame_files = sorted([
    f for f in os.listdir(FRAMES_DIR)
    if f.lower().endswith((".jpg", ".png"))
])

print(f"Total frames found: {len(frame_files)}")

people_counts = []
frame_indices = []

# processing frames
for idx, frame_name in enumerate(frame_files):
    frame_path = os.path.join(FRAMES_DIR, frame_name)
    frame = cv2.imread(frame_path)

    if frame is None:
        continue

    results = model(frame)[0]

    people_count = 0
    if results.boxes is not None:
        for cls, conf in zip(results.boxes.cls, results.boxes.conf):
            if int(cls) == 0 and conf >= CONF_THRESHOLD:
                people_count += 1

    people_counts.append(people_count)
    frame_indices.append(idx)

    print(f"Frame {idx + 1}/{len(frame_files)} People detected: {people_count}")

df = pd.DataFrame({
    "frame": frame_indices,
    "y": people_counts
})

df["ds"] = pd.to_datetime(df["frame"], unit="s")
df = df[["ds", "y"]]

df.to_csv(
    os.path.join(DATA_DIR, "people_count_v6.csv"),
    index=False
)

# adding seasonality
prophet_model = Prophet(
    daily_seasonality=False,
    weekly_seasonality=False,
    yearly_seasonality=False
)

prophet_model.add_seasonality(
    name="frame_pattern",
    period=150,
    fourier_order=5
)

prophet_model.fit(df)

future = prophet_model.make_future_dataframe(
    periods=FUTURE_FRAMES,
    freq="s"
)

forecast = prophet_model.predict(future)

forecast_minutes = forecast[["ds", "yhat"]].copy()
forecast_minutes["minute"] = (
    (forecast_minutes["ds"] - forecast_minutes["ds"].iloc[0])
    .dt.total_seconds() // 60
).astype(int)

forecast_minutes = (
    forecast_minutes
    .groupby("minute", as_index=False)
    .mean()
)

forecast_minutes["predicted_people"] = np.floor(forecast_minutes["yhat"].clip(lower=0)).astype(int)
forecast_minutes = forecast_minutes[forecast_minutes["minute"] <= 133]
forecast_minutes = forecast_minutes[["minute", "predicted_people"]]

forecast_minutes.to_csv(
    os.path.join(DATA_DIR, "people_forecast_v6.csv"),
    index=False
)

df_minutes = df.copy()
df_minutes["minute"] = (
    (df_minutes["ds"] - df_minutes["ds"].iloc[0])
    .dt.total_seconds() // 60
).astype(int)

df_minutes = (
    df_minutes
    .groupby("minute", as_index=False)
    .mean()
)

df_minutes = df_minutes[df_minutes["minute"] <= 133]
df_minutes.rename(columns={"y": "observed_people"}, inplace=True)

df["minute"] = (df["ds"] - df["ds"].iloc[0]).dt.total_seconds() // 60
df["minute"] = df["minute"].astype(int)

people_per_minute = (
    df.groupby("minute", as_index=False)["y"]
    .mean()
)

people_per_minute["people_count"] = np.floor(people_per_minute["y"].clip(lower=0)).astype(int)
people_per_minute = people_per_minute[people_per_minute["minute"] <= 133]
people_per_minute = people_per_minute[["minute", "people_count"]]

people_per_minute["people_count"] = np.floor(
    people_per_minute["people_count"].clip(lower=0)
).astype(int)

people_per_minute.to_csv(
    os.path.join(DATA_DIR, "people_count_v6.csv"),
    index=False
)

forecast["yhat"] = np.floor(forecast["yhat"].clip(lower=0)).astype(int)
forecast["yhat_lower"] = np.floor(forecast["yhat_lower"].clip(lower=0)).astype(int)
forecast["yhat_upper"] = np.floor(forecast["yhat_upper"].clip(lower=0)).astype(int)

plt.figure(figsize=(16, 8))

# actual count
actual_minutes = people_per_minute["minute"]
actual_counts = people_per_minute["people_count"]

# light visual smoothing
actual_smooth = (
    pd.Series(actual_counts)
    .rolling(window=5, center=True, min_periods=1)
    .mean()
)

plt.plot(
    actual_minutes,
    actual_smooth,
    color="black",
    linewidth=2,
    label="Observed people count"
)

# forecast
last_actual_minute = actual_minutes.max()

forecast_minutes_shifted = (
    forecast_minutes["minute"] + last_actual_minute
)

forecast_smooth = (
    pd.Series(forecast_minutes["predicted_people"])
    .rolling(window=5, center=True, min_periods=1)
    .mean()
)

plt.plot(
    forecast_minutes_shifted,
    forecast_smooth,
    color="blue",
    linewidth=2,
    label="Forecast"
)

# confidence band
plt.fill_between(
    forecast_minutes_shifted,
    forecast_smooth * 0.85,
    forecast_smooth * 1.15,
    color="blue",
    alpha=0.25,
    label="Confidence interval"
)

# axes
plt.xlim(0, last_actual_minute + 133)
plt.xticks([20, 40, 60, 80, 100, 120, last_actual_minute, last_actual_minute + 133])
plt.ylim(bottom=0)

plt.xlabel("Time (minutes)")
plt.ylabel("Number of people")
plt.title("People count over time with frame based forecast")

plt.legend()
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()

plt.savefig(os.path.join(GRAPH_DIR, "count_forecast_v6.png"))
plt.close()

print("Done")
print(f"Csv saved in: {DATA_DIR}")
print(f"Graph saved in: {GRAPH_DIR}")

Total frames found: 4540

0: 480x640 4 potted plants, 104.6ms
Speed: 24.9ms preprocess, 104.6ms inference, 21.4ms postprocess per image at shape (1, 3, 480, 640)
Frame 1/4540 People detected: 0

0: 480x640 4 potted plants, 47.5ms
Speed: 1.3ms preprocess, 47.5ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)
Frame 2/4540 People detected: 0

0: 480x640 4 potted plants, 47.0ms
Speed: 1.1ms preprocess, 47.0ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Frame 3/4540 People detected: 0

0: 480x640 5 potted plants, 45.9ms
Speed: 1.2ms preprocess, 45.9ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)
Frame 4/4540 People detected: 0

0: 480x640 4 potted plants, 46.0ms
Speed: 1.2ms preprocess, 46.0ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Frame 5/4540 People detected: 0

0: 480x640 4 potted plants, 45.3ms
Speed: 1.3ms preprocess, 45.3ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)
Frame 6/4540 

23:41:46 - cmdstanpy - INFO - Chain [1] start processing
23:41:47 - cmdstanpy - INFO - Chain [1] done processing


Done
Csv saved in: ../data/new_data
Graph saved in: ../visualization/graphs
